Restarted venv (Python 3.11.7)

 # Phase 4: Unified Router & Pipeline Evaluation
 This interactive notebook/script evaluates the Phase 4 hybrid architecture:
 - **Step 4.1**: Setup and Environment Initialization
 - **Step 4.2**: Query Router Test (Small-scale inspection on individual query types)
 - **Step 4.3**: Query Router Full Audit (Classification metrics across all ground-truth questions)
 - **Step 4.4**: Pipeline Single Query Walkthrough (Detailed inspection of SQL path vs RAG path outputs)
 - **Step 4.5**: Full Pipeline Evaluation (Hit Rate@5 and MRR@5 across ground truth dataset)

In [ ]:
# Cell 1: Setup & Environment
import os
import pandas as pd
from sentence_transformers import SentenceTransformer

from src import config
from src.db import get_db_connection
from src.router import classify_query
from src.sql_engine import text_to_sql_answer
from src.rag import rag_answer
from src.pipeline import answer_query

# Load ground-truth dataset
GROUND_TRUTH_PATH = os.path.join("data", "ground_truth.csv")
if not os.path.exists(GROUND_TRUTH_PATH):
    raise FileNotFoundError(f"Missing '{GROUND_TRUTH_PATH}'. Please run 01_foundations.py first!")

df_gt = pd.read_csv(GROUND_TRUTH_PATH)
print(f"✅ Loaded {len(df_gt)} ground-truth questions for evaluation.")

# Initialize embedding model and DB connection
print("Loading embedding model 'all-MiniLM-L6-v2'...")
model = SentenceTransformer("all-MiniLM-L6-v2")
conn = get_db_connection()
print("✅ Database connection and embedding model ready.")

c:\Users\pnala\Desktop\IDP_Archive\idp_codebase\Digital_Archive_Research\rag_dev\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 150 ground-truth questions for evaluation.
Loading embedding model 'all-MiniLM-L6-v2'...
✅ Database connection and embedding model ready.


 ## Step 4.2: Query Router Test (Small Scale Inspection)
 Test `classify_query()` incrementally on a few sample questions to verify classification rules before running bulk evaluation.

In [ ]:
# Cell 2: Small Scale Router Test
test_queries = [
    "What is the billing date for invoice INV-2026-0145?",
    "What was the total amount spent on Laptops last month?",
    "Show me all invoices from Summit Hardware",
    "Which vendor issued invoice INV-2026-0130?",
    "I bought some cleaning supplies around March, can you find which invoice?"
]

print("--- Small-Scale Query Classification Test ---")
for q in test_queries:
    method = classify_query(q)
    print(f"Query  : '{q}'")
    print(f"Routed : [{method.upper()}]\n")

--- Small-Scale Query Classification Test ---
Query  : 'What is the billing date for invoice INV-2026-0145?'
Routed : [SQL]

Query  : 'What was the total amount spent on Laptops last month?'
Routed : [SQL]

Query  : 'Show me all invoices from Summit Hardware'
Routed : [SQL]

Query  : 'Which vendor issued invoice INV-2026-0130?'
Routed : [SQL]

Query  : 'I bought some cleaning supplies around March, can you find which invoice?'
Routed : [SQL]



 ## Step 4.3: Query Router Audit Across Dataset
 Run the router on all ground-truth questions and break down classification by `question_type`.

In [ ]:
# Cell 3: Full Router Audit
df_gt["routed_method"] = df_gt["question"].apply(classify_query)

sql_count = (df_gt["routed_method"] == "sql").sum()
rag_count = (df_gt["routed_method"] == "rag").sum()

print("=== ROUTER CLASSIFICATION AUDIT ===")
print(f"Total Questions        : {len(df_gt)}")
print(f"Routed to SQL Path     : {sql_count} ({sql_count/len(df_gt)*100:.1f}%)")
print(f"Routed to RAG Path     : {rag_count} ({rag_count/len(df_gt)*100:.1f}%)\n")

print("--- Classification Breakdown by Question Type ---")
type_breakdown = df_gt.groupby(["question_type", "routed_method"]).size().unstack(fill_value=0)
print(type_breakdown)

=== ROUTER CLASSIFICATION AUDIT ===
Total Questions        : 150
Routed to SQL Path     : 150 (100.0%)
Routed to RAG Path     : 0 (0.0%)

--- Classification Breakdown by Question Type ---
routed_method      sql
question_type         
count_aggregation    1
date_lookup         28
line_item_price     24
line_item_qty       28
subtotal_lookup     29
total_lookup        19
vendor_lookup       21


 ## Step 4.4: Single Query Pipeline Walkthrough
 Inspect pipeline execution outputs for one SQL query and one RAG query.

In [ ]:
# Cell 4: Single Query Inspection
# 1. SQL Path Inspection
sql_sample = df_gt[df_gt["routed_method"] == "sql"].iloc[0]
print("--- SQL Path Walkthrough ---")
print(f"Question : {sql_sample['question']}")
print(f"Expected : {sql_sample['expected_answer']} (Invoice: {sql_sample['source_invoice_id']})")

sql_res = answer_query(sql_sample["question"], model, conn)
print(f"Method   : {sql_res['method'].upper()}")
print(f"SQL Query: {sql_res.get('sql')}")
print(f"Answer   : {sql_res.get('answer')}\n")

# 2. RAG Path Inspection
rag_sample = df_gt[df_gt["routed_method"] == "rag"].iloc[0] if rag_count > 0 else df_gt.iloc[0]
print("--- RAG Path Walkthrough ---")
print(f"Question : {rag_sample['question']}")
print(f"Expected : {rag_sample['expected_answer']} (Invoice: {rag_sample['source_invoice_id']})")

rag_res = answer_query(rag_sample["question"], model, conn, force_method="rag")
print(f"Method   : {rag_res['method'].upper()}")
print(f"Top Hit  : {rag_res['retrieved_chunks'][0]['invoice_id'] if rag_res.get('retrieved_chunks') else 'None'}")
print(f"Answer   : {rag_res.get('answer')}\n")

--- SQL Path Walkthrough ---
Question : What is the billing date for invoice INV-2026-0145?
Expected : 2026-05-03 (Invoice: INV-2026-0145)
Method   : SQL
SQL Query: SELECT content_json FROM invoice_chunks WHERE content_json->>'invoice_id' = 'INV-2026-0145';
Answer   : The billing date for invoice **INV-2026-0145** is **May 3, 2026**.

--- RAG Path Walkthrough ---
Question : What is the billing date for invoice INV-2026-0145?
Expected : 2026-05-03 (Invoice: INV-2026-0145)
Method   : RAG
Top Hit  : INV-2025-0071
Answer   : I could not find this in the available invoices. (Invoice ID: INV-2026-0145)



 ## Step 4.5: Full Pipeline Retrieval Evaluation
 Evaluate Hit Rate@5 and MRR@5 across all ground-truth queries using the unified router pipeline.

In [ ]:
# Cell 5: Bulk Pipeline Benchmark
hits = 0
mrr_sum = 0.0

print("Running Full Pipeline Retrieval Benchmark...")

for idx, row in df_gt.iterrows():
    q = row["question"]
    expected_id = row["source_invoice_id"]
    
    res = answer_query(q, model, conn)
    
    found = False
    rank = 0

    if res["method"] == "sql":
        # Check if expected_id is present in SQL query data result or generated SQL string
        data_str = str(res.get("data", "")) + str(res.get("sql", ""))
        if expected_id in data_str:
            found = True
            rank = 1
    else:
        # Check if expected_id is present in retrieved chunks
        chunks = res.get("retrieved_chunks") or []
        for r_idx, chunk in enumerate(chunks, 1):
            if chunk.get("invoice_id") == expected_id:
                found = True
                rank = r_idx
                break

    if found:
        hits += 1
        mrr_sum += 1.0 / rank

hit_rate = (hits / len(df_gt)) * 100
mrr = mrr_sum / len(df_gt)

print("\n=== UNIFIED PIPELINE EVALUATION RESULTS ===")
print(f"Total Queries Evaluated : {len(df_gt)}")
print(f"Hits                    : {hits}")
print(f"Hit Rate @ 5 (%)        : {hit_rate:.2f}%")
print(f"MRR @ 5                 : {mrr:.4f}")

# Close database connection
conn.close()

Running Full Pipeline Retrieval Benchmark...

=== UNIFIED PIPELINE EVALUATION RESULTS ===
Total Queries Evaluated : 150
Hits                    : 149
Hit Rate @ 5 (%)        : 99.33%
MRR @ 5                 : 0.9933
